#Install Library

In [54]:
pip install requests pillow

#Import Library

In [65]:
import base64
import json
import requests
import io
import sqlite3
from PIL import Image, ImageEnhance
from datetime import datetime

#SET API KEY

In [56]:
OPENROUTER_API_KEY = (
    "masukkan api keys kalian"
)

#Image Classification Using AI

In [57]:
def encode_image_to_base64(image_path: str) -> str:
    """Fungsi untuk mengubah file gambar lokal menjadi format Base64."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [58]:
def classify_document(
    image_path: str, api_key: str = OPENROUTER_API_KEY
) -> dict:
    """Mengklasifikasikan apakah gambar merupakan KTP Indonesia.

    Mencoba beberapa kandidat model Vision secara otomatis (Fallback Mechanism).
    """
    base64_image = encode_image_to_base64(image_path)
    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-Title": "KTP Classification App",
    }

    prompt_text = (
        "Apakah gambar ini merupakan KTP Indonesia?\n"
        "Jawab HANYA dalam format JSON seperti contoh berikut tanpa teks lain:\n"
        '{"is_ktp": true}\n'
        "atau\n"
        '{"is_ktp": false}'
    )

    # Daftar kandidat model Vision OpenRouter (Diuji secara berurutan)
    candidate_models = [
        "google/gemini-2.0-flash-exp:free",
        "meta-llama/llama-3.2-11b-vision-instruct:free",
        "openai/gpt-4o-mini",
        "qwen/qwen-2-vl-7b-instruct:free",
    ]

    for model_name in candidate_models:
        print(f"🔄 Mencoba model: {model_name}...")

        payload = {
            "model": model_name,
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt_text},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            },
                        },
                    ],
                }
            ],
            "response_format": {"type": "json_object"},
        }

        try:
            response = requests.post(
                url, headers=headers, json=payload, timeout=30
            )

            if response.status_code == 200:
                result = response.json()
                raw_content = result["choices"][0]["message"]["content"]
                print(f"✅ Berhasil merespons menggunakan model: {model_name}\n")
                return json.loads(raw_content)
            else:
                print(
                    f"⚠️ Model {model_name} gagal [{response.status_code}]: {response.text}"
                )

        except Exception as e:
            print(f"⚠️ Exception pada model {model_name}: {e}")

    print("\n❌ Semua kandidat model gagal merespons.")
    return {"is_ktp": False}

##Testing

In [59]:
if __name__ == "__main__":
    # Ganti dengan path gambar KTP / Non-KTP milikmu
    test_image_path = "ktp.jpeg"

    print("Memproses Klasifikasi Gambar...")
    classification = classify_document(test_image_path)

    print("\nHasil Klasifikasi:")
    print(json.dumps(classification, indent=2))

    # Logika Percabangan Sesuai Workflow Proyek
    if classification.get("is_ktp"):
        print("\n✅ Dokumen terdeteksi sebagai KTP Indonesia.")
        print("Lanjutkan ke proses OCR Extraction...")
    else:
        print("\n🛑 STOP PROCESS: Gambar bukan KTP Indonesia!")

Memproses Klasifikasi Gambar...
🔄 Mencoba model: google/gemini-2.0-flash-exp:free...
⚠️ Model google/gemini-2.0-flash-exp:free gagal [404]: {"error":{"message":"No endpoints found for google/gemini-2.0-flash-exp:free.","code":404},"user_id":"user_3Dye6F2CdFurc9OOumljr00Hbir"}
🔄 Mencoba model: meta-llama/llama-3.2-11b-vision-instruct:free...
⚠️ Model meta-llama/llama-3.2-11b-vision-instruct:free gagal [404]: {"error":{"message":"No endpoints found for meta-llama/llama-3.2-11b-vision-instruct:free.","code":404},"user_id":"user_3Dye6F2CdFurc9OOumljr00Hbir"}
🔄 Mencoba model: openai/gpt-4o-mini...
✅ Berhasil merespons menggunakan model: openai/gpt-4o-mini


Hasil Klasifikasi:
{
  "is_ktp": true
}

✅ Dokumen terdeteksi sebagai KTP Indonesia.
Lanjutkan ke proses OCR Extraction...


#OCR Extraction Using AI

In [60]:
def preprocess_and_encode_image(image_path: str) -> str:
    """Melakukan preprocessing gambar (Upscale + Enhancement)

    agar piksel teks NIK jauh lebih jelas untuk AI Vision.
    """
    img = Image.open(image_path)

    # Convert ke RGB jika gambar berpola RGBA/PNG
    if img.mode in ("RGBA", "P"):
        img = img.convert("RGB")

    # 1. Perbesar ukuran gambar 2x lipat dengan filter Lanczos
    w, h = img.size
    img = img.resize((w * 2, h * 2), Image.Resampling.LANCZOS)

    # 2. Pertajam kontras teks gambar
    enhancer = ImageEnhance.Contrast(img)
    img = enhancer.enhance(1.4)

    # Convert ke Base64
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=95)
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

In [61]:
def extract_ktp_ocr(
    image_path: str, api_key: str = OPENROUTER_API_KEY
) -> dict:
    """Mengekstrak data KTP secara presisi menggunakan Image Preprocessing & Cross-Verification Prompting."""
    base64_image = preprocess_and_encode_image(image_path)
    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-Title": "KTP OCR Extraction App",
    }

    # Prompt dengan instruksi Cross-Verification
    prompt_text = """
    Kamu adalah sistem OCR KTP Indonesia tingkat presisi tinggi.
    Ekstrak seluruh data dari gambar KTP ini ke dalam JSON dengan sangat teliti.

    ATURAN VERIFIKASI KETAT UNTUK NIK (16 DIGIT):
    1. NIK Wajib terdiri dari TEPAT 16 DIGIT ANGKA.
    2. Struktur NIK KTP Indonesia:
       - Digit 1-6 : Kode Wilayah
       - Digit 7-12: Tanggal Lahir (DDMMYY).
         * Catatan: Jika Wanita, Tanggal Lahir pada NIK ditambahkan 40.
       - Digit 13-16: Nomor Urut Registrasi
    3. Lakukan double-check pada Digit 7-12 NIK: Pastikan angkanya SAMA KONSISTEN dengan nilai 'tempat_tgl_lahir' dan 'jenis_kelamin' yang kamu baca di gambar KTP tersebut. Jangan sampai salah membaca digit '0' atau '2'.

    Format Kembalikan HANYA JSON persis seperti berikut:
    {
        "nik": "",
        "nama": "",
        "tempat_tgl_lahir": "",
        "jenis_kelamin": "",
        "agama": "",
        "alamat": "",
        "rt": "",
        "rw": "",
        "kelurahan": "",
        "kecamatan": "",
        "status_perkawinan": "",
        "pekerjaan": "",
        "kewarganegaraan": "",
        "berlaku_hingga": ""
    }
    """

    candidate_models = [
        "openai/gpt-4o",  # Model utama resolusi tinggi
        "openai/gpt-4o-mini",  # Fallback
    ]

    for model_name in candidate_models:
        payload = {
            "model": model_name,
            "temperature": 0.0,
            "messages": [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt_text},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}",
                                "detail": "high",
                            },
                        },
                    ],
                }
            ],
            "response_format": {"type": "json_object"},
        }

        try:
            response = requests.post(
                url, headers=headers, json=payload, timeout=30
            )

            if response.status_code == 200:
                result = response.json()
                raw_content = result["choices"][0]["message"]["content"]
                return json.loads(raw_content)

        except Exception as e:
            print(f"⚠️ Kendala pada model {model_name}: {e}")

    return {}

##Testing

In [62]:
if __name__ == "__main__":
    test_image_path = "ktp.jpeg"

    print("🔍 Menjalankan Ekstraksi OCR menggunakan AI Vision...")
    ocr_result = extract_ktp_ocr(test_image_path)

    print("\n📄 Hasil Ekstraksi JSON:")
    print(json.dumps(ocr_result, indent=4, ensure_ascii=False))

🔍 Menjalankan Ekstraksi OCR menggunakan AI Vision...

📄 Hasil Ekstraksi JSON:
{
    "nik": "3471XXXXXXXXXXXX",
    "nama": "DEWI TXX LXXXXXX",
    "tempat_tgl_lahir": "BANTUL, XX-XX-XXXX",
    "jenis_kelamin": "PEREMPUAN",
    "agama": "ISLAM",
    "alamat": "SEMAKI GEDE XX X/XXX",
    "rt": "XXX",
    "rw": "XXX",
    "kelurahan": "SEMAKI",
    "kecamatan": "UMBULHARJO",
    "status_perkawinan": "BELUM KAWIN",
    "pekerjaan": "PELAJAR/MAHASISWA",
    "kewarganegaraan": "WNI",
    "berlaku_hingga": "SEUMUR HIDUP"
}


#Business Rule Validation

In [63]:
def validate_ktp_business_rules(data: dict) -> dict:
    validation_summary = {
        "nik_length": {"valid": False, "message": ""},
        "nik_is_numeric": {"valid": False, "message": ""},
        "dob_format": {"valid": False, "message": ""},
        "gender_match": {"valid": False, "message": ""},
        "dob_match": {"valid": False, "message": ""},
        "overall_status": "INVALID",
    }

    nik = str(data.get("nik", "")).strip()
    tempat_tgl_lahir = str(data.get("tempat_tgl_lahir", "")).strip()
    jenis_kelamin = str(data.get("jenis_kelamin", "")).strip().upper()

    validation_summary["nik_is_numeric"]["valid"] = nik.isdigit()
    validation_summary["nik_is_numeric"][
        "message"
    ] = "NIK berisi angka." if nik.isdigit() else "NIK mengandung non-angka."

    validation_summary["nik_length"]["valid"] = len(nik) == 16
    validation_summary["nik_length"][
        "message"
    ] = "Panjang NIK 16 digit." if len(nik) == 16 else f"Panjang NIK {len(nik)} digit."

    tgl_str = (
        tempat_tgl_lahir.split(",")[-1].strip()
        if "," in tempat_tgl_lahir
        else tempat_tgl_lahir
    )
    parsed_dob = None
    try:
        parsed_dob = datetime.strptime(tgl_str, "%d-%m-%Y")
        validation_summary["dob_format"]["valid"] = True
        validation_summary["dob_format"][
            "message"
        ] = f"Format tanggal valid ({tgl_str})."
    except ValueError:
        validation_summary["dob_format"][
            "message"
        ] = f"Format tanggal '{tgl_str}' salah."

    if len(nik) == 16 and nik.isdigit() and parsed_dob:
        nik_day, nik_month, nik_year = (
            int(nik[6:8]),
            int(nik[8:10]),
            int(nik[10:12]),
        )
        is_female = nik_day > 40
        expected_gender = "PEREMPUAN" if is_female else "LAKI-LAKI"

        validation_summary["gender_match"]["valid"] = (
            jenis_kelamin == expected_gender
        )
        validation_summary["gender_match"][
            "message"
        ] = f"Jenis kelamin '{jenis_kelamin}' sesuai NIK."

        actual_nik_day = nik_day - 40 if is_female else nik_day
        dob_match = (
            actual_nik_day == parsed_dob.day
            and nik_month == parsed_dob.month
            and nik_year == (parsed_dob.year % 100)
        )
        validation_summary["dob_match"]["valid"] = dob_match
        validation_summary["dob_match"][
            "message"
        ] = "Tanggal lahir di NIK cocok dengan tempat_tgl_lahir."

    all_passed = all(
        v["valid"]
        for k, v in validation_summary.items()
        if k != "overall_status"
    )
    if all_passed:
        validation_summary["overall_status"] = "VALID"

    return validation_summary

##Testing

In [64]:
if __name__ == "__main__":
    test_image_path = "ktp.jpeg"

    print("🛡️ Menjalankan Validasi Business Rule secara otomatis...")
    print("⏳ Mengambil data OCR langsung dari gambar...")

    # Memanggil fungsi OCR dari cell sebelumnya tanpa import & tanpa hardcode manual
    ocr_data = extract_ktp_ocr(test_image_path)

    print("\n🔍 Memvalidasi data hasil ekstraksi OCR...")
    validation_res = validate_ktp_business_rules(ocr_data)

    print(
        f"\n📊 STATUS AKHIR VALIDASI: {validation_res['overall_status']}\n"
    )
    print(json.dumps(validation_res, indent=4, ensure_ascii=False))

🛡️ Menjalankan Validasi Business Rule secara otomatis...
⏳ Mengambil data OCR langsung dari gambar...

🔍 Memvalidasi data hasil ekstraksi OCR...

📊 STATUS AKHIR VALIDASI: VALID

{
    "nik_length": {
        "valid": true,
        "message": "Panjang NIK 16 digit."
    },
    "nik_is_numeric": {
        "valid": true,
        "message": "NIK berisi angka."
    },
    "dob_format": {
        "valid": true,
        "message": "Format tanggal valid (30-07-2002)."
    },
    "gender_match": {
        "valid": true,
        "message": "Jenis kelamin 'PEREMPUAN' sesuai NIK."
    },
    "dob_match": {
        "valid": true,
        "message": "Tanggal lahir di NIK cocok dengan tempat_tgl_lahir."
    },
    "overall_status": "VALID"
}


#Database

In [66]:
def init_db(db_name: str = "ktp_database.db"):
    """Inisialisasi database SQLite dan membuat tabel ktp_records jika belum ada."""
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS ktp_records (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nik TEXT,
        nama TEXT,
        tempat_tgl_lahir TEXT,
        jenis_kelamin TEXT,
        agama TEXT,
        alamat TEXT,
        rt TEXT,
        rw TEXT,
        kelurahan TEXT,
        kecamatan TEXT,
        status_perkawinan TEXT,
        pekerjaan TEXT,
        kewarganegaraan TEXT,
        berlaku_hingga TEXT,
        overall_status TEXT,
        validation_details TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)

    conn.commit()
    conn.close()

In [67]:
def save_ktp_record(
    ocr_data: dict, validation_res: dict, db_name: str = "ktp_database.db"
) -> int:
    """Menyimpan data hasil OCR dan ringkasan validasi ke database SQLite.

    Returns:
        int: ID record data yang baru tersimpan.
    """
    # Pastikan database & tabel sudah siap
    init_db(db_name)

    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    cursor.execute(
        """
    INSERT INTO ktp_records (
        nik, nama, tempat_tgl_lahir, jenis_kelamin, agama, alamat,
        rt, rw, kelurahan, kecamatan, status_perkawinan, pekerjaan,
        kewarganegaraan, berlaku_hingga, overall_status, validation_details
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """,
        (
            ocr_data.get("nik", ""),
            ocr_data.get("nama", ""),
            ocr_data.get("tempat_tgl_lahir", ""),
            ocr_data.get("jenis_kelamin", ""),
            ocr_data.get("agama", ""),
            ocr_data.get("alamat", ""),
            ocr_data.get("rt", ""),
            ocr_data.get("rw", ""),
            ocr_data.get("kelurahan", ""),
            ocr_data.get("kecamatan", ""),
            ocr_data.get("status_perkawinan", ""),
            ocr_data.get("pekerjaan", ""),
            ocr_data.get("kewarganegaraan", ""),
            ocr_data.get("berlaku_hingga", ""),
            validation_res.get("overall_status", "INVALID"),
            json.dumps(validation_res, ensure_ascii=False),
        ),
    )

    record_id = cursor.lastrowid
    conn.commit()
    conn.close()
    return record_id

In [68]:
def get_all_ktp_records(db_name: str = "ktp_database.db") -> list:
    """Membaca seluruh riwayat data KTP tersimpan dari database."""
    conn = sqlite3.connect(db_name)
    conn.row_factory = sqlite3.Row  # Agar output berupa dictionary per baris
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM ktp_records ORDER BY created_at DESC")
    rows = cursor.fetchall()

    results = [dict(row) for row in rows]
    conn.close()
    return results

##Testing

In [69]:
if __name__ == "__main__":
    test_image_path = "ktp.jpeg"

    print("💾 [TAHAP 4] Menjalankan Integrasi Penyimpanan Database SQLite...")

    # 1. Menjalankan OCR secara otomatis dari gambar
    print("⏳ [1/3] Memproses OCR Extraction...")
    ocr_data = extract_ktp_ocr(test_image_path)

    # 2. Menjalankan Validasi Business Rules
    print("⏳ [2/3] Memproses Business Rule Validation...")
    validation_res = validate_ktp_business_rules(ocr_data)

    # 3. Menyimpan hasil ke SQLite Database
    print("⏳ [3/3] Menyimpan Data ke Database SQLite...")
    inserted_id = save_ktp_record(ocr_data, validation_res)
    print(f"✅ Data KTP berhasil disimpan dengan Record ID: {inserted_id}")

    # 4. Verifikasi dengan membaca ulang isi database
    print("\n📄 Verifikasi Data yang Tersimpan di Database:")
    all_data = get_all_ktp_records()
    print(json.dumps(all_data, indent=4, ensure_ascii=False))

💾 [TAHAP 4] Menjalankan Integrasi Penyimpanan Database SQLite...
⏳ [1/3] Memproses OCR Extraction...
⏳ [2/3] Memproses Business Rule Validation...
⏳ [3/3] Menyimpan Data ke Database SQLite...
✅ Data KTP berhasil disimpan dengan Record ID: 1

📄 Verifikasi Data yang Tersimpan di Database:
[
    {
        "id": 1,
        "nik": "3471XXXXXXXXXXXX",
        "nama": "DEWI TXX LXXXXXX",
        "tempat_tgl_lahir": "BANTUL, XX-XX-XXXX",
        "jenis_kelamin": "PEREMPUAN",
        "agama": "ISLAM",
        "alamat": "SEMAKI GEDE XX X/XXX",
        "rt": "XXX",
        "rw": "XXX",
        "kelurahan": "SEMAKI",
        "kecamatan": "UMBULHARJO",
        "status_perkawinan": "BELUM KAWIN",
        "pekerjaan": "PELAJAR/MAHASISWA",
        "kewarganegaraan": "WNI",
        "berlaku_hingga": "SEUMUR HIDUP",
        "overall_status": "VALID",
        "validation_details": "{\"nik_length\": {\"valid\": true, \"message\": \"Panjang NIK 16 digit.\"}, \"nik_is_numeric\": {\"valid\": true, \"message\